# S7.3 — Chat API

Interactive verification of the chat endpoint with session management,
follow-up detection, streaming SSE, and citation-backed responses.

In [1]:
import sys
sys.path.insert(0, "../..")

from src.schemas.api.chat import ChatRequest, ChatResponse, SessionHistoryResponse, SessionClearResponse
print("\u2713 Chat schema imports successful")

✓ Chat schema imports successful


## FR-1: ChatRequest Validation

In [2]:
# Valid request
req = ChatRequest(session_id="sess-1", query="What is a transformer?")
assert req.session_id == "sess-1"
assert req.stream is True
print(f"Valid request: session_id={req.session_id}, stream={req.stream}")

# Query stripping
req2 = ChatRequest(session_id="s1", query="  hello  ")
assert req2.query == "hello"
print(f"Query stripped: '{req2.query}'")

# Validation errors
from pydantic import ValidationError
try:
    ChatRequest(session_id="", query="hi")
    assert False, "Should have raised"
except ValidationError:
    print("Empty session_id correctly rejected")

try:
    ChatRequest(session_id="s1", query="   ")
    assert False, "Should have raised"
except ValidationError:
    print("Whitespace query correctly rejected")

print("\n\u2713 FR-1: ChatRequest validation works correctly")

Valid request: session_id=sess-1, stream=True
Query stripped: 'hello'
Empty session_id correctly rejected
Whitespace query correctly rejected

✓ FR-1: ChatRequest validation works correctly


## FR-2: Chat Endpoint (JSON mode)

In [3]:
import httpx
from httpx import ASGITransport, AsyncClient
from unittest.mock import AsyncMock
from src.main import create_app
from src.dependency import get_follow_up_handler, get_conversation_memory
from src.services.chat.follow_up import FollowUpResult
from src.services.rag.models import RAGResponse, SourceReference

# Create mock handler
mock_handler = AsyncMock()
mock_handler.handle = AsyncMock(return_value=FollowUpResult(
    original_query="What is attention?",
    rewritten_query="What is attention?",
    is_follow_up=False,
    response=RAGResponse(
        answer="Attention is a mechanism [1].",
        sources=[SourceReference(
            index=1,
            arxiv_id="1706.03762",
            title="Attention Is All You Need",
            authors=["Vaswani et al."],
            arxiv_url="https://arxiv.org/abs/1706.03762",
        )],
        query="What is attention?",
    ),
))

app = create_app()
app.dependency_overrides[get_follow_up_handler] = lambda: mock_handler
app.dependency_overrides[get_conversation_memory] = lambda: None

transport = ASGITransport(app=app)
async with AsyncClient(transport=transport, base_url="http://test") as client:
    resp = await client.post("/api/v1/chat", json={
        "session_id": "demo-1",
        "query": "What is attention?",
        "stream": False,
    })

assert resp.status_code == 200
data = resp.json()
print(f"Status: {resp.status_code}")
print(f"Answer: {data['answer']}")
print(f"Sources: {len(data['sources'])} paper(s)")
print(f"Session: {data['session_id']}")
print(f"Follow-up: {data['is_follow_up']}")
assert data["session_id"] == "demo-1"
assert len(data["sources"]) == 1
print("\n\u2713 FR-2: JSON mode works correctly")

Status: 200
Answer: Attention is a mechanism [1].
Sources: 1 paper(s)
Session: demo-1
Follow-up: False

✓ FR-2: JSON mode works correctly


## FR-2: Chat Endpoint (Streaming mode)

In [4]:
from collections.abc import AsyncIterator

async def mock_stream():
    yield "Transformers are "
    yield "powerful [1]."

mock_handler2 = AsyncMock()
mock_handler2.handle_stream = AsyncMock(return_value=mock_stream())
mock_handler2._memory = None

app2 = create_app()
app2.dependency_overrides[get_follow_up_handler] = lambda: mock_handler2
app2.dependency_overrides[get_conversation_memory] = lambda: None

transport2 = ASGITransport(app=app2)
async with AsyncClient(transport=transport2, base_url="http://test") as client:
    resp = await client.post("/api/v1/chat", json={
        "session_id": "demo-stream",
        "query": "What are transformers?",
    })

body = resp.text
print("SSE events received:")
for line in body.split("\n"):
    if line.startswith("event:"):
        print(f"  {line}")

assert "event: metadata" in body
assert "event: token" in body
assert "event: done" in body
print("\n\u2713 FR-2: Streaming mode works correctly")

SSE events received:
  event: metadata
  event: token
  event: token
  event: sources
  event: done

✓ FR-2: Streaming mode works correctly


## FR-3: Session Management

In [5]:
from src.services.chat.memory import ChatMessage

mock_memory = AsyncMock()
mock_memory.get_history = AsyncMock(return_value=[
    ChatMessage(role="user", content="What is attention?"),
    ChatMessage(role="assistant", content="Attention is a mechanism [1]."),
])
mock_memory.clear_session = AsyncMock(return_value=True)

app3 = create_app()
app3.dependency_overrides[get_follow_up_handler] = lambda: mock_handler
app3.dependency_overrides[get_conversation_memory] = lambda: mock_memory

transport3 = ASGITransport(app=app3)
async with AsyncClient(transport=transport3, base_url="http://test") as client:
    # Get history
    resp = await client.get("/api/v1/chat/sessions/demo-1/history")
    assert resp.status_code == 200
    data = resp.json()
    print(f"History: {len(data['messages'])} messages")
    for msg in data["messages"]:
        print(f"  [{msg['role']}] {msg['content'][:50]}")
    
    # Clear session
    resp = await client.delete("/api/v1/chat/sessions/demo-1")
    assert resp.status_code == 200
    assert resp.json()["cleared"] is True
    print(f"\nSession cleared: {resp.json()['cleared']}")

print("\n\u2713 FR-3: Session management works correctly")

History: 2 messages
  [user] What is attention?
  [assistant] Attention is a mechanism [1].

Session cleared: True

✓ FR-3: Session management works correctly


## Summary

S7.3 Chat API provides:
- `POST /api/v1/chat` — conversational Q&A with follow-up detection
- Streaming SSE mode (metadata → tokens → sources → done)
- JSON mode for non-streaming responses
- `GET /api/v1/chat/sessions/{id}/history` — retrieve history
- `DELETE /api/v1/chat/sessions/{id}` — clear session
- Graceful degradation when Redis is unavailable